# Energy Transition & Electric Mobility in Portugal - Data Extraction

**Phase 1: Extraction**

Pulls REN daily production/consumption data and E-REDES electric mobility
consumption data (aggregated by year/month/district/municipality).

In [3]:
import requests
import json
import time
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path

# VS Code Jupyter runs notebooks with the working directory set to the folder
# containing the .ipynb file. This notebook lives in notebooks/, so the
# project root is exactly one level up — no marker file, no searching.
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

REN_BASE_URL = "https://servicebus.ren.pt/datahubapi/electricity"
EREDES_BASE_URL = "https://e-redes.opendatasoft.com/api/explore/v2.1/catalog/datasets"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data will be saved to: {DATA_RAW}")

Project root: c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics
Data will be saved to: c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\raw


## REN - daily production/consumption breakdown

A single call per day already returns every source (Hydro, Solar, Wind,
Natural Gas, Biomass, Coal, Wave, Import, Export, batteries, etc.) as
separate series, at 15-minute resolution.

In [ ]:
def fetch_ren_daily_breakdown(target_date, delay=0.5):
    """Fetch REN electricity production breakdown for a single date."""
    params = {"culture": "en-US", "date": target_date.isoformat()}
    response = requests.get(
        f"{REN_BASE_URL}/ElectricityProductionBreakdownDaily", params=params
    )
    time.sleep(delay)

    if response.status_code != 200:
        print(f"Failed for {target_date}: {response.status_code}")
        return None

    return response.json()


def build_date_range(start_date, end_date):
    days = (end_date - start_date).days
    return [start_date + timedelta(days=i) for i in range(days + 1)]


END_DATE = date.today() - timedelta(days=1)
START_DATE = END_DATE - relativedelta(years=3)
date_range = build_date_range(START_DATE, END_DATE)

print(f"Fetching {len(date_range)} days ({START_DATE} to {END_DATE})...")

ren_raw_results = []
ren_failed_dates = []

for d in date_range:
    result = fetch_ren_daily_breakdown(d)
    if result is not None:
        ren_raw_results.append({"date": d, "data": result})
    else:
        ren_failed_dates.append(d)

print(f"Success: {len(ren_raw_results)} | Failed: {len(ren_failed_dates)}")

output_path = DATA_RAW / "ren_production_breakdown_raw.json"
with open(output_path, "w") as f:
    json.dump(ren_raw_results, f, indent=2, default=str)

print(f"Saved to {output_path}")

NameError: name 'date' is not defined

## E-REDES - electric mobility consumption, aggregated

Two datasets cover all 18 mainland districts between them:
`_01_a_10` and `_11_a_18`. Aggregated server-side (sum of energy per
year/month/district/municipality) instead of pulling raw hourly rows.

`total_count` is unreliable on `group_by` queries here - pagination stops
when a page comes back empty, not by trusting `total_count`.

In [ ]:
def fetch_eredes_grouped(dataset_id, page_size=200):
    """Fetch all grouped/aggregated records, paginating until an empty page."""
    all_results = []
    offset = 0

    while True:
        response = requests.get(
            f"{EREDES_BASE_URL}/{dataset_id}/records",
            params={
                "select": "sum(energia_consumida) as total_energia_kwh",
                "group_by": "ano,mes,distrito,concelho",
                "limit": page_size,
                "offset": offset,
            },
        )
        response.raise_for_status()
        results = response.json()["results"]

        if not results:
            break

        all_results.extend(results)
        offset += page_size
        time.sleep(0.3)

    return all_results


EREDES_DATASETS = [
    "consumo_horario_mobilidade_eletrica_01_a_10",
    "consumo_horario_mobilidade_eletrica_11_a_18",
]

eredes_raw_results = []
for dataset_id in EREDES_DATASETS:
    print(f"Fetching {dataset_id}...")
    rows = fetch_eredes_grouped(dataset_id)
    print(f"  {len(rows)} grouped rows")
    eredes_raw_results.extend(rows)

print(f"Total rows: {len(eredes_raw_results)}")

output_path = DATA_RAW / "eredes_mobility_grouped_raw.json"
with open(output_path, "w") as f:
    json.dump(eredes_raw_results, f, indent=2, default=str)

print(f"Saved to {output_path}")

## Next: 02_cleaning.ipynb

- Flatten the REN series (xAxis.categories + series[].data) into a tidy long-format table
- Convert E-REDES grouped rows into a clean DataFrame, check for duplicate district/municipality names across the two source datasets